In [28]:
import pandas as pd
import os
from itertools import permutations

## Unión de clorofila y reflectancias

In [13]:
# Cargamos los csv de los tifs
path = "saved_files/"
dfs_tifs = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_tiffs_") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_tifs[nombre_sin_extension] = pd.read_csv(ruta_completa)

# Y de las boyas
path = "saved_files/"
dfs_boyas = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_boyas_") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_boyas[nombre_sin_extension] = pd.read_csv(ruta_completa)

In [14]:
band_names = {
    "Band_1": "rhow_B1",
    "Band_2": "rhow_B2",
    "Band_3": "rhow_B3",
    "Band_4": "rhow_B4",
    "Band_5": "rhow_B5",
    "Band_6": "rhow_B6",
    "Band_7": "rhow_B7",
    "Band_8": "rhow_B8",
    "Band_9": "rhown_B1",
    "Band_10": "rhown_B2",
    "Band_11": "rhown_B3",
    "Band_12": "rhown_B4",
    "Band_13": "rhown_B5",
    "Band_14": "rhown_B6",
}
for nombre_df, df in dfs_tifs.items():
    dfs_tifs[nombre_df] = df.rename(columns=band_names)


In [15]:
dfs_tifs["df_tiffs_c2x-complex-nets_1x1"].head(3)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,rhow_B7,rhow_B8,rhown_B1,rhown_B2,rhown_B3,rhown_B4,rhown_B5,rhown_B6,Band_15,Band_16
0,2017-06-30,CTD1,4187246,695025,0.023568,0.034184,0.043664,0.014988,0.009146,0.002320,0.002351,0.000930,0.023582,0.034339,0.043720,0.015355,0.009758,0.002110,0.001174,-8.000000e-45
1,2017-06-30,CTD2,4181518,693105,0.013951,0.022364,0.026365,0.006800,0.003868,0.001000,0.001059,0.000427,0.014022,0.022509,0.026334,0.007046,0.004299,0.000942,0.000236,-8.000000e-45
2,2017-06-30,CTD3,4181698,695238,0.017592,0.024643,0.031816,0.010574,0.007355,0.001946,0.001922,0.000778,0.017462,0.024656,0.031839,0.010682,0.007440,0.001862,0.104799,-8.000000e-45


In [17]:
dfs_boyas["df_boyas_upct_depth_gt_1"].head(3)

,Date,Buoy,Chl
0,2017-05-19,CTD1,0.883
1,2017-05-19,CTD10,0.837
2,2017-05-19,CTD11,0.899


In [26]:
merge_dict = {}

for df_tif_name, df_tif in dfs_tifs.items():
    for df_boya_name, df_boya in dfs_boyas.items():
        print(df_tif_name[9:], df_boya_name[9:])
        merge_dict[f"{df_tif_name[9:]}_{df_boya_name[9:]}"] = df_tif.merge(df_boya, how="inner", on=["Date", "Buoy"])    

c2x-nets_5x5 upct_depth_gt_1
c2x-nets_5x5 imida_depth_gt_1
c2x-nets_5x5 imida_depth_lt_2
c2x-nets_5x5 upct_depth_lt_2
c2x-nets_5x5 upct_depth_lt_1
c2x-nets_5x5 imida_depth_lt_1
c2x-nets_3x3 upct_depth_gt_1
c2x-nets_3x3 imida_depth_gt_1
c2x-nets_3x3 imida_depth_lt_2
c2x-nets_3x3 upct_depth_lt_2
c2x-nets_3x3 upct_depth_lt_1
c2x-nets_3x3 imida_depth_lt_1
c2x-complex-nets_5x5 upct_depth_gt_1
c2x-complex-nets_5x5 imida_depth_gt_1
c2x-complex-nets_5x5 imida_depth_lt_2
c2x-complex-nets_5x5 upct_depth_lt_2
c2x-complex-nets_5x5 upct_depth_lt_1
c2x-complex-nets_5x5 imida_depth_lt_1
c2x-complex-nets_1x1 upct_depth_gt_1
c2x-complex-nets_1x1 imida_depth_gt_1
c2x-complex-nets_1x1 imida_depth_lt_2
c2x-complex-nets_1x1 upct_depth_lt_2
c2x-complex-nets_1x1 upct_depth_lt_1
c2x-complex-nets_1x1 imida_depth_lt_1
c2x-complex-nets_9x9 upct_depth_gt_1
c2x-complex-nets_9x9 imida_depth_gt_1
c2x-complex-nets_9x9 imida_depth_lt_2
c2x-complex-nets_9x9 upct_depth_lt_2
c2x-complex-nets_9x9 upct_depth_lt_1
c2x-compl

In [27]:
for df_name, df in merge_dict.items():
    df.to_csv(f"saved_files/dataset/{df_name}.csv", index=False)

## Creación de features

In [30]:
# Diferencia normalizada

def diferencia_normalizada(band1, band2):
    value = (band1 - band2)/(band1 + band2)
    return value

def dall_gitelson(band1, band2, band3):
    value = (1/(band1) - 1/(band2))*(band3)
    return value

def diferencia_normalizada_4bandas(band1, band2, band3, band4):
    value = (band1 - band2)/(band3 + band4)
    return value

def diferencia_inversas(band1, band2):
    value = 1/(band1) - 1/(band2)
    return value

def diferencia_relacion_4bandas(band1, band2, band3, band4):
    value = band1/band2 - band3/band4
    return value


In [36]:

index_list = []
def add_two_band_difs(data):
    spectral_bands = ['rhow_B1', 'rhow_B2', 'rhow_B3', 'rhow_B4']
    for i, band1 in enumerate(spectral_bands):
        for band2 in spectral_bands[i+1:]:
            df[f"dif_norm_{band1}_{band2}"] = diferencia_normalizada(data[band1], data[band2])
            index_list.append(f"dif_norm_{band1}_{band2}")
            df[f"dif_inv_{band1}_{band2}"] = diferencia_inversas(data[band1], data[band2])
            index_list.append(f"dif_inv_{band1}_{band2}")
add_two_band_difs(df)



In [39]:
# VER CÓMO EVITAR RELACIONES SIMÉTRICAS

index_dall_gitelson_list = []
def add_dall_gitelson(data):
    spectral_bands = ['rhow_B1', 'rhow_B2', 'rhow_B3', 'rhow_B4']
    for band1, band2, band3 in permutations(spectral_bands, 3):  
        df[f"dall_gitelson_{band1}_{band2}_{band3}"] = dall_gitelson(data[band1], data[band2], data[band3])
        index_dall_gitelson_list.append(f"dall_gitelson_{band1}_{band2}_{band3}")
add_dall_gitelson(df)


In [40]:
index_dall_gitelson_list

['dall_gitelson_rhow_B1_rhow_B2_rhow_B3',
 'dall_gitelson_rhow_B1_rhow_B2_rhow_B4',
 'dall_gitelson_rhow_B1_rhow_B3_rhow_B2',
 'dall_gitelson_rhow_B1_rhow_B3_rhow_B4',
 'dall_gitelson_rhow_B1_rhow_B4_rhow_B2',
 'dall_gitelson_rhow_B1_rhow_B4_rhow_B3',
 'dall_gitelson_rhow_B2_rhow_B1_rhow_B3',
 'dall_gitelson_rhow_B2_rhow_B1_rhow_B4',
 'dall_gitelson_rhow_B2_rhow_B3_rhow_B1',
 'dall_gitelson_rhow_B2_rhow_B3_rhow_B4',
 'dall_gitelson_rhow_B2_rhow_B4_rhow_B1',
 'dall_gitelson_rhow_B2_rhow_B4_rhow_B3',
 'dall_gitelson_rhow_B3_rhow_B1_rhow_B2',
 'dall_gitelson_rhow_B3_rhow_B1_rhow_B4',
 'dall_gitelson_rhow_B3_rhow_B2_rhow_B1',
 'dall_gitelson_rhow_B3_rhow_B2_rhow_B4',
 'dall_gitelson_rhow_B3_rhow_B4_rhow_B1',
 'dall_gitelson_rhow_B3_rhow_B4_rhow_B2',
 'dall_gitelson_rhow_B4_rhow_B1_rhow_B2',
 'dall_gitelson_rhow_B4_rhow_B1_rhow_B3',
 'dall_gitelson_rhow_B4_rhow_B2_rhow_B1',
 'dall_gitelson_rhow_B4_rhow_B2_rhow_B3',
 'dall_gitelson_rhow_B4_rhow_B3_rhow_B1',
 'dall_gitelson_rhow_B4_rhow_B3_rh

In [ ]:



index_4_bands_list = []
def add_index_4_bands(data):
    spectral_bands = ['UltraBlue', 'Blue', 'Green', 'Red', 'NIR1']
    for band1, band2, band3, band4 in permutations(spectral_bands, 4):
        df[f"index_4_bands_{band1}_{band2}_{band3}_{band4}"] = index_4_bands(
            data[band1], data[band2], data[band3], data[band4])
        index_4_bands_list.append(f"index_4_bands_{band1}_{band2}_{band3}_{band4}")
add_index_4_bands(df)


two_div_differences_list = []
def add_two_div_differences(data):
    spectral_bands = ['UltraBlue', 'Blue', 'Green', 'Red', 'NIR1']
    for band1, band2, band3, band4 in permutations(spectral_bands, 4):
        df[f"two_div_dif_{band1}_{band2}_{band3}_{band4}"] = data[band1]/data[band2] - data[band3]/data[band4]
        two_div_differences_list.append(f"two_div_dif_{band1}_{band2}_{band3}_{band4}")
add_two_div_differences(df)